In [1]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

/home/timothytoonen/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [22]:
ham_df = pd.read_csv("../data/raw/HAM10000/HAM10000_metadata.csv")
pad_df = pd.read_csv("../data/raw/PAD_UFES_20/metadata.csv")
bcn_df = pd.read_csv("../data/raw/BCN20000/bcn_20k_train.csv")

In [23]:
ham_df

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear
...,...,...,...,...,...,...,...
10010,HAM_0002867,ISIC_0033084,akiec,histo,40.0,male,abdomen
10011,HAM_0002867,ISIC_0033550,akiec,histo,40.0,male,abdomen
10012,HAM_0002867,ISIC_0033536,akiec,histo,40.0,male,abdomen
10013,HAM_0000239,ISIC_0032854,akiec,histo,80.0,male,face


In [24]:
pad_df

,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,skin_cancer_history,...,diameter_2,diagnostic,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed
0,PAT_1516,1765,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,...,NaN,NEV,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1516_1765_530.png,False
1,PAT_46,881,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,5.0,BCC,TRUE,TRUE,FALSE,TRUE,TRUE,TRUE,PAT_46_881_939.png,True
2,PAT_1545,1867,NaN,NaN,NaN,NaN,77,NaN,NaN,NaN,...,NaN,ACK,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1545_1867_547.png,False
3,PAT_1989,4061,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,...,NaN,ACK,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1989_4061_934.png,False
4,PAT_684,1302,False,True,POMERANIA,POMERANIA,79,False,MALE,True,...,5.0,BCC,TRUE,TRUE,FALSE,FALSE,TRUE,TRUE,PAT_684_1302_588.png,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2293,PAT_1708,3156,NaN,NaN,NaN,NaN,73,NaN,NaN,NaN,...,NaN,ACK,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1708_3156_175.png,False
2294,PAT_46,880,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,12.0,BCC,TRUE,TRUE,FALSE,TRUE,FALSE,FALSE,PAT_46_880_140.png,True
2295,PAT_1343,1217,NaN,NaN,NaN,NaN,74,NaN,NaN,NaN,...,NaN,SEK,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1343_1217_404.png,False
2296,PAT_326,690,False,False,POMERANIA,POMERANIA,58,True,FEMALE,True,...,4.0,BCC,TRUE,FALSE,FALSE,FALSE,FALSE,TRUE,PAT_326_690_823.png,True


In [25]:
bcn_df

,bcn_filename,age_approx,anatom_site_general,diagnosis,lesion_id,capture_date,sex,split
0,BCN_0000000001.jpg,55.0,anterior torso,MEL,BCN_0003884,2012-05-16,male,train
1,BCN_0000000003.jpg,50.0,anterior torso,MEL,BCN_0000019,2015-07-09,female,train
2,BCN_0000000004.jpg,85.0,head/neck,SCC,BCN_0003499,2015-11-23,male,train
3,BCN_0000000006.jpg,60.0,anterior torso,NV,BCN_0003316,2015-06-16,male,train
4,BCN_0000000010.jpg,30.0,anterior torso,BCC,BCN_0004874,2014-02-18,female,train
...,...,...,...,...,...,...,...,...
12408,BCN_0000020348.jpg,85.0,head/neck,BCC,BCN_0003925,2013-03-05,female,train
12409,BCN_0000020349.jpg,65.0,anterior torso,BKL,BCN_0001819,2016-05-05,male,train
12410,BCN_0000020350.jpg,70.0,lower extremity,MEL,BCN_0001085,2015-01-29,male,train
12411,BCN_0000020352.jpg,55.0,palms/soles,NV,BCN_0002083,2016-05-08,female,train


In [26]:
# Standardize across all dataframes

ham_clean = pd.DataFrame({
    'image_id': ham_df['image_id'] + '.jpg', # jpg missing
    'raw_label': ham_df['dx'].astype(str).str.lower().str.strip(),
    'domain': 'HAM10000'
})

pad_clean = pd.DataFrame({
    'image_id': pad_df['img_id'],
    'raw_label': pad_df['diagnostic'].astype(str).str.lower().str.strip(),
    'domain': 'PAD_UFES_20'
})

bcn_clean = pd.DataFrame({
    'image_id': bcn_df['bcn_filename'],
    'raw_label': bcn_df['diagnosis'].astype(str).str.lower().str.strip(),
    'domain': 'BCN20000'
})

In [29]:
main_df = pd.concat([ham_clean, pad_clean, bcn_clean], ignore_index=True)


# Due to issues later on, must map
class_map = {
    'mel': 'MEL', 'melanoma': 'MEL',
    'nv': 'NV', 'nev': 'NV', 'nevus': 'NV',
    'bcc': 'BCC', 'basal cell carcinoma': 'BCC',
    'bkl': 'BKL', 'sek': 'BKL', 'seborrheic keratosis': 'BKL',
    'akiec': 'AKIEC', 'ack': 'AKIEC', 'scc': 'AKIEC', 'ak': 'AKIEC', 'actinic keratosis': 'AKIEC'
}

main_df['label'] = main_df['raw_label'].map(class_map)


unmapped_labels = main_df[main_df['label'].isna()]['raw_label'].unique()
if len(unmapped_labels) > 0:
    print(f"WARNING: The following raw medical labels were not mapped and will be dropped: {unmapped_labels}")
    
main_df = main_df.dropna(subset=['label'])
main_df = main_df.drop(columns=['raw_label'])


# Split out of distr data
source_df = main_df[main_df['domain'].isin(['HAM10000', 'BCN20000'])].copy()
target_df = main_df[main_df['domain'] == 'PAD_UFES_20'].copy()
target_df['split'] = 'test'

In [30]:
train_df, val_df = train_test_split(
    source_df, 
    test_size=0.10, 
    stratify=source_df['label'], 
    random_state=42
)

train_df['split'] = 'train'
val_df['split'] = 'val'

In [31]:
# Save final metadata

final_df = pd.concat([train_df, val_df, target_df])
final_df.to_csv("../data/raw/master_metadata_split.csv", index=False)

print("\n--- Domain Split Completed ---")
print("TRAIN (Clinical Only):", len(train_df))
print("VAL (Clinical Only):", len(val_df))
print("TEST (Smartphone Only):", len(target_df))


--- Domain Split Completed ---
TRAIN (Clinical Only): 19742
VAL (Clinical Only): 2194
TEST (Smartphone Only): 2298
